In [1]:
%load_ext autoreload
%autoreload 2

# Customer Churn Prediction

A telecom company is losing customers to competitors. The sales and marketing team wants to know **which customers are likely to leave next month**, so they can offer them an incentive to stay.

- **Data:** [Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn) (7,043 customers)
- **Target:** `Churn` (Yes/No)

The logic lives in the `telco_churn` package (`src/telco_churn/`). This notebook calls it and explains the decisions.

In [2]:
import pandas as pd

from telco_churn.data import TARGET, clean_data, load_raw_data, split_features_target

## 1. Loading and cleaning the data

In [3]:
raw = load_raw_data()
raw.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Column types

In [4]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

`TotalCharges` should be a number, but it was read as text (`str`). That means at least one value in the column can't be parsed as a number.

In [5]:
total_numeric = pd.to_numeric(raw["TotalCharges"], errors="coerce")
unparseable = total_numeric.isna()

print(f"Rows that are not a number: {unparseable.sum()}")
print(f"Their values: {raw.loc[unparseable, 'TotalCharges'].unique().tolist()}")
raw.loc[unparseable, ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]

Rows that are not a number: 11
Their values: [' ']


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


In [6]:
# Are these exactly the customers with tenure == 0?
new_customers = raw["tenure"] == 0
print(f"Customers with tenure == 0: {new_customers.sum()}")
print(f"Same rows as the blanks: {(new_customers == unparseable).all()}")

Customers with tenure == 0: 11
Same rows as the blanks: True


All 11 blanks are a single space `" "`, and they are exactly the 11 customers with `tenure == 0`. These are **new customers who have not been billed yet**, so their true total charge is **0**. It is not an unknown value.

That's why we fill with 0 instead of dropping the rows or filling with an average:
- dropping would discard real (if brand-new) customers;
- the mean or median would invent charges these customers never paid.

Only these 11 values are not numbers, so `errors="coerce"` is safe here.

### Number of distinct values per column

In [7]:
raw.nunique()

customerID          7043
gender                 2
SeniorCitizen          2
Partner                2
Dependents             2
tenure                73
PhoneService           2
MultipleLines          3
InternetService        3
OnlineSecurity         3
OnlineBackup           3
DeviceProtection       3
TechSupport            3
StreamingTV            3
StreamingMovies        3
Contract               3
PaperlessBilling       2
PaymentMethod          4
MonthlyCharges      1585
TotalCharges        6531
Churn                  2
dtype: int64

- `customerID` is different for every row (7,043 values). It says nothing about churn and would only let a model memorise individual customers, so we **drop** it.
- `SeniorCitizen` has 2 values, but is stored as 0/1 while the other yes/no columns (`Partner`, `Dependents`, …) use "No"/"Yes". We convert it to "No"/"Yes" so it's encoded like the other categories and not scaled as a number.
- `tenure`, `MonthlyCharges` and `TotalCharges` are the only real numeric columns.
- All other columns are categories with 2–4 values.

### Cleaning

`clean_data` applies these decisions to a copy of the data (`raw` is unchanged):

| Column | Change | Reason |
|---|---|---|
| `TotalCharges` | text → number, blanks → 0 | tenure 0, not billed yet |
| `SeniorCitizen` | 0/1 → "No"/"Yes" | same form as other yes/no columns |
| `Churn` | "No"/"Yes" → 0/1 | scikit-learn treats 1 as the positive class; churn rate = mean |
| `customerID` | dropped | identifier, not a feature |

In [8]:
df = clean_data(raw)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   str    
 1   SeniorCitizen     7043 non-null   str    
 2   Partner           7043 non-null   str    
 3   Dependents        7043 non-null   str    
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   str    
 6   MultipleLines     7043 non-null   str    
 7   InternetService   7043 non-null   str    
 8   OnlineSecurity    7043 non-null   str    
 9   OnlineBackup      7043 non-null   str    
 10  DeviceProtection  7043 non-null   str    
 11  TechSupport       7043 non-null   str    
 12  StreamingTV       7043 non-null   str    
 13  StreamingMovies   7043 non-null   str    
 14  Contract          7043 non-null   str    
 15  PaperlessBilling  7043 non-null   str    
 16  PaymentMethod     7043 non-null   str    
 17  Monthl

In [9]:
assert df.isna().sum().sum() == 0, "cleaning left missing values"

X, y = split_features_target(df)
print(f"X: {X.shape}, y: {y.shape}")
print(f"Churn rate: {y.mean():.1%}")

X: (7043, 19), y: (7043,)
Churn rate: 26.5%
